# IndicOOC — Fine-tuning multilingual SigLIP (Objective O3, first pass)

Notebooks 01, 04 and 05 all use frozen encoders. This one adapts an encoder to the task.

**What is trained and what is not.** The vision tower is frozen. LoRA adapters are attached to the *text* tower only, plus the logit scale and bias. Roughly 1–2M trainable parameters against ~200M frozen ones.

That is a hypothesis-driven choice, not a compute shortcut. Notebook 05 exists to test whether the text side is the weak link for Indic script. If it is, adapting the text tower is the targeted fix; if the vision tower were also unfrozen, a gain could not be attributed to either. Freezing it keeps the experiment interpretable.

**Why LoRA rather than a full fine-tune.** At 1–2k examples a full fine-tune of 200M parameters memorises the training set within a couple of epochs and degrades the pretrained multilingual representations that make the model useful in the first place. LoRA constrains the update to a low-rank subspace, which is the standard mitigation at this data scale.

**Silver trains, gold evaluates.** If your training rows come from machine translation, they are silver-standard: the labels are inherited, the phrasing is synthetic. Never report a test number on silver data. Train on silver, tune on silver validation, and report only on your human-annotated pilot rows. Mixing the two makes every number unquotable.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q transformers accelerate peft pillow pandas scikit-learn sentencepiece

## Step 1 — Config

`TRAIN_CSV` and `VAL_CSV` must have the same columns as `pilot_v0.csv`: `id`, `claim_text`, `image_url`, `label`, `language`. Labels must be `supported` or `out_of_context`.

`GOLD_TEST_CSV` is your human-annotated held-out split. It is never trained on.

In [ ]:
import os

PROJECT_ROOT = '/content/drive/MyDrive/IndicOOC'
BASE_MODEL   = 'google/siglip-base-patch16-256-multilingual'

TRAIN_CSV     = f'{PROJECT_ROOT}/data/annotations/silver_train.csv'
VAL_CSV       = f'{PROJECT_ROOT}/data/annotations/silver_val.csv'
GOLD_TEST_CSV = f'{PROJECT_ROOT}/data/annotations/splits/test.csv'

IMAGE_CACHE = '/content/image_cache'
ADAPTER_DIR = f'{PROJECT_ROOT}/models/siglip_multi_lora'

EPOCHS         = 4
BATCH_SIZE     = 8
LEARNING_RATE  = 1e-4
LORA_R         = 16
LORA_ALPHA     = 32
LORA_DROPOUT   = 0.1
MAX_TEXT_LEN   = 64

os.makedirs(IMAGE_CACHE, exist_ok=True)
os.makedirs(ADAPTER_DIR, exist_ok=True)

## Step 2 — Cache every image to local disk

Downloading inside the training loop would re-fetch the same image once per epoch and make each epoch hostage to network flakiness. Fetch once, cache to `/content`, and let failed rows drop out with a count printed.

Expect this cell to be the slowest in the notebook on a few thousand rows.

In [ ]:
import hashlib, requests, pandas as pd
from PIL import Image
from io import BytesIO

def cache_path(url):
    return os.path.join(IMAGE_CACHE, hashlib.md5(url.encode()).hexdigest() + '.jpg')

def ensure_cached(url, timeout=15):
    p = cache_path(url)
    if os.path.exists(p):
        return p
    try:
        r = requests.get(url, timeout=timeout, headers={'User-Agent': 'Mozilla/5.0'})
        r.raise_for_status()
        Image.open(BytesIO(r.content)).convert('RGB').save(p, 'JPEG', quality=92)
        return p
    except Exception:
        return None

def prepare(csv_path, name):
    df = pd.read_csv(csv_path)
    df['local_path'] = [ensure_cached(u) for u in df['image_url']]
    before = len(df)
    df = df[df['local_path'].notna()].copy()
    print(f'{name}: {len(df)}/{before} rows usable')
    return df

train_df = prepare(TRAIN_CSV, 'train')
val_df   = prepare(VAL_CSV, 'val')
print(train_df['label'].value_counts())

## Step 3 — Dataset

Each row becomes one (image, claim, target) triple where target is 1.0 for `supported` and 0.0 for `out_of_context`.

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoProcessor, AutoModel

processor = AutoProcessor.from_pretrained(BASE_MODEL)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

class PairDataset(Dataset):
    def __init__(self, df):
        self.rows = df.reset_index(drop=True)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        r = self.rows.iloc[i]
        return {
            'image': Image.open(r['local_path']).convert('RGB'),
            'text': str(r['claim_text']),
            'target': 1.0 if r['label'] == 'supported' else 0.0,
        }

def collate(batch):
    enc = processor(
        text=[b['text'] for b in batch],
        images=[b['image'] for b in batch],
        padding='max_length',
        truncation=True,
        max_length=MAX_TEXT_LEN,
        return_tensors='pt',
    )
    enc['targets'] = torch.tensor([b['target'] for b in batch], dtype=torch.float)
    return enc

train_loader = DataLoader(PairDataset(train_df), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)
val_loader   = DataLoader(PairDataset(val_df), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate)
print(len(train_loader), 'train batches')

## Step 4 — Attach LoRA to the text tower only

Target modules are discovered by name rather than hardcoded, so this survives a checkpoint whose internal naming differs. The filter keeps only modules under `text_model`, which is what freezes the vision side.

In [ ]:
from peft import LoraConfig, get_peft_model

model = AutoModel.from_pretrained(BASE_MODEL)

targets = sorted({
    name for name, mod in model.named_modules()
    if name.startswith('text_model')
    and isinstance(mod, torch.nn.Linear)
    and name.split('.')[-1] in {'q_proj', 'k_proj', 'v_proj', 'out_proj', 'fc1', 'fc2'}
})
print(f'{len(targets)} text-tower modules targeted')

config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=targets,
    bias='none',
)

model = get_peft_model(model, config)
for n, p in model.named_parameters():
    if 'logit_scale' in n or 'logit_bias' in n:
        p.requires_grad = True
model.to(device)
model.print_trainable_parameters()

## Step 5 — Train

The loss is binary cross-entropy on the diagonal of `logits_per_image` — the logit for each image paired with *its own* claim. This matches how the model is used at inference: one pair scored on its own, not ranked against a batch.

`pos_weight` compensates if the training set is class-imbalanced.

In [ ]:
import numpy as np
from torch.nn import BCEWithLogitsLoss

n_pos = float((train_df['label'] == 'supported').sum())
n_neg = float((train_df['label'] == 'out_of_context').sum())
pos_weight = torch.tensor([n_neg / max(n_pos, 1.0)], device=device)
loss_fn = BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LEARNING_RATE)

def run_epoch(loader, train):
    model.train() if train else model.eval()
    total, n = 0.0, 0
    preds, golds = [], []
    for batch in loader:
        targets = batch.pop('targets').to(device)
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.set_grad_enabled(train):
            out = model(**batch)
            logits = out.logits_per_image.diagonal()
            loss = loss_fn(logits, targets)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
        total += loss.item() * len(targets)
        n += len(targets)
        preds += (torch.sigmoid(logits) > 0.5).long().cpu().tolist()
        golds += targets.long().cpu().tolist()
    acc = float(np.mean(np.array(preds) == np.array(golds)))
    return total / max(n, 1), acc

best = -1.0
for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(train_loader, True)
    va_loss, va_acc = run_epoch(val_loader, False)
    print(f'epoch {epoch}: train loss {tr_loss:.4f} acc {tr_acc:.3f} | val loss {va_loss:.4f} acc {va_acc:.3f}')
    if va_acc > best:
        best = va_acc
        model.save_pretrained(ADAPTER_DIR)
        print(f'  saved (best val acc {best:.3f})')

## Step 6 — Record what was trained

`scoring.py` reads this file to know which base checkpoint the adapter belongs to. An adapter loaded onto the wrong base produces silently wrong scores rather than an error, so this is worth writing.

In [ ]:
import json

meta = {
    'base_model': BASE_MODEL,
    'trained_rows': int(len(train_df)),
    'val_rows': int(len(val_df)),
    'epochs': EPOCHS,
    'lora_r': LORA_R,
    'best_val_accuracy': round(best, 4),
    'frozen': 'vision tower',
}
with open(os.path.join(ADAPTER_DIR, 'indicooc_meta.json'), 'w') as f:
    json.dump(meta, f, indent=2)
meta

## Step 7 — Evaluate on the gold pilot rows

This is the only number worth quoting. The rows here were annotated by hand and never appeared in training.

In [ ]:
from sklearn.metrics import classification_report

gold = prepare(GOLD_TEST_CSV, 'gold test')
gold_loader = DataLoader(PairDataset(gold), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate)

model.eval()
scores = []
for batch in gold_loader:
    batch.pop('targets')
    batch = {k: v.to(device) for k, v in batch.items()}
    with torch.no_grad():
        logits = model(**batch).logits_per_image.diagonal()
    scores += torch.sigmoid(logits).cpu().tolist()

gold = gold.copy()
gold['score'] = scores
gold['pred_label'] = ['supported' if s > 0.5 else 'out_of_context' for s in scores]

print(classification_report(gold['label'], gold['pred_label'], zero_division=0))
gold[['id', 'language', 'label', 'pred_label', 'score']]

## How to read the result

If the gold test split is still small, this report is a smoke test of the training path, not evidence of a gain. A fine-tuned model that scores well on a handful of rows has told you almost nothing.

What this notebook does establish:

- An adaptation path exists end to end: data in, LoRA out, adapter saved, served by the app
- The training and evaluation sets are separated by provenance, so a silver/gold leak cannot creep in unnoticed
- The comparison against frozen encoders in notebook 05 runs on identical gold rows, so the two are directly comparable

The claim to make is *"adaptation is implemented and measurable"*. The claim to avoid is *"fine-tuning improves performance"* until the gold test split is large enough to support it.